# Project - Statistical arbitrage in cryptocurrencies 
For this project I have used the data from binance and have chosen 11 cryptos from 2021. The analysis is made with 1 hour data to test the momentum strategy and reversal strategy. 

In [1]:
from binance.client import Client as bnb_client
from datetime import datetime
import pandas as pd 
import numpy as np 
import random
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA, IncrementalPCA

client = bnb_client()

def get_binance_px(symbol,freq,start_ts = '2021-01-01'):
    data = client.get_historical_klines(symbol,freq,start_ts)
    columns = ['open_time','open','high','low','close','volume','close_time','quote_volume',
    'num_trades','taker_base_volume','taker_quote_volume','ignore']

    data = pd.DataFrame(data,columns = columns)
    
    # Convert from POSIX timestamp (number of millisecond since jan 1, 1970)
    data['open_time'] = data['open_time'].map(lambda x: datetime.utcfromtimestamp(x/1000))
    data['close_time'] = data['close_time'].map(lambda x: datetime.utcfromtimestamp(x/1000))
    return data 
#['BTC-USD','ETH-USD','BNB-USD','ADA-USD','LINK-USD','XRP-USD','XLM-USD','DOGE-USD','AVAX-USD']
univ = ['BTCUSDT','ETHUSDT','ADAUSDT','BNBUSDT','XRPUSDT','DOTUSDT','MATICUSDT','LINKUSDT', 'XLMUSDT','DOGEUSDT','AVAXUSDT']

freq = '1h'
open_data = 1 #0 to read from pickle file 1 for binance package

if open_data == 0:
    px = {}
    for x in univ:
        data = get_binance_px(x,freq)
        px[x] = data.set_index('open_time')['close']

    px = pd.DataFrame(px).astype(float)
    px = px.reindex(pd.date_range(px.index[0],px.index[-1],freq=freq))
    px.to_pickle('crypto_px_1h.pk')
    
else: 
    px = pd.read_pickle('crypto_px_1h.pk')
ret = px.pct_change()

/var/folders/08/r2cys9850vg1d2qf5mp1pq0w0000gn/T/ipykernel_22652/3886405217.py:41: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret = px.pct_change()


The nan in the data is dropped using pandas function and then a plot of the cumulative returns is plotted below

In [2]:
#from binance.client import Client

#api_key = "xxx"
#api_secret = "xxx"

# client = Client(api_key, api_secret)
exchange_info = bnb_client.get_exchange_info()
print(exchange_info)

#for s in exchange_info['symbols']:
#    print(s['symbol'])

TypeError: Client.get_exchange_info() missing 1 required positional argument: 'self'

In [ ]:
ret.dropna()

In [3]:
univ

['BTCUSDT',
 'ETHUSDT',
 'ADAUSDT',
 'BNBUSDT',
 'XRPUSDT',
 'DOTUSDT',
 'MATICUSDT',
 'LINKUSDT',
 'XLMUSDT',
 'DOGEUSDT',
 'AVAXUSDT']

In [ ]:
ret.cumsum().plot()
sample_data = {}
sample_data['day of the week'] = ret.index.dayofweek

### A scatter plot of the returns is plotted against the day of the week
The results dont show any vast grouping preferring one day to the other, this is a cude visualization check to see for any specific scatter of the group of cryptos in the data. 

In [ ]:
ret.plot(
    kind='scatter',
    x=ret.index.dayofweek, 
    y=univ, 
    backend='plotly', 
)

## Defining and testing a momentum strategy

The 

In [ ]:
# Getting the momentum strategy 
import math

def compute_momentum(ret, window_size):
   momentum = ret.rolling(window_size).mean()/ret.rolling(window_size).std()*math.sqrt(window_size)
   return momentum

def compute_portfolio(momentum):
    portfolio = (abs(momentum) > 1.0)*1.0
    portfolio = portfolio.div(portfolio.abs().sum(1),0)
    portfolio = np.tanh(portfolio)
    return portfolio
momentum = compute_momentum(ret,31*24*3)
portfolio_m = compute_portfolio(momentum)


In [ ]:
strat_ret = (portfolio_m.shift()*ret).sum(1)

In [ ]:
strat_ret.cumsum().plot()

In [ ]:
# sharpe
strat_ret.mean()/strat_ret.std()*math.sqrt(365*24)

In [ ]:
# sharpe within each year

sharpe = lambda x: x.mean()/x.std()*math.sqrt(365*24) 
strat_ret.groupby([x.year for x in strat_ret.index]).apply(sharpe)

In [ ]:
ret.shape

In [ ]:
count = lambda x: x.shape 
strat_ret.groupby([x.year for x in strat_ret.index]).apply(count)

In [ ]:
def compute_mom_method(ret,hor):
    avg_ret = ret.rolling(hor,min_periods=1).mean().rank(1)
    avg_ret = avg_ret.subtract(avg_ret.mean(1),0)
    avg_ret = avg_ret.divide(avg_ret.abs().sum(1),0)
    avg_ret = np.tanh(avg_ret)
    return(avg_ret)

strats = {}
xx = 24*365
list_iter = list(range(1,60))
my_new_list = [i * 24 for i in list_iter]
for hor in my_new_list:
    avg_ret = compute_mom_method(ret,hor)
    # momentum_ret = compute_momentum(ret,hor)
    # avg_ret = compute_portfolio(momentum_ret)
    strats[hor] = (avg_ret.shift()*ret).sum(1)
strats = pd.DataFrame(strats)

In [ ]:
sr = strats.mean()/strats.std()*np.sqrt(365*24)

In [ ]:
sr.max()

In [ ]:
plt.plot(sr)

In [ ]:
ret.cumsum().plot()

In [ ]:
ret.corrwith(strat_ret)

In [ ]:
avg_ret.plot()

In [ ]:
strats[240].mean()/strats[240].std()*np.sqrt(365*24)

In [ ]:
strats.groupby([x.year for x in strat_ret.index]).apply(sharpe).T.plot()

In [ ]:
# Initialize the top car manufacturing companies into a list (using their ticker symbols):

# Initialize a DataFrame to store the p-values:
p_values = pd.DataFrame(index=univ, columns=univ)

In [ ]:
import itertools
# Perform a pairwise Engle-Granger test on each possible pair of stocks in the list:

stock_data = px.dropna()

for pair in itertools.combinations(univ, 2):
    # Regress one stock on the other:
    Y = stock_data[pair[0]].dropna()
    X = stock_data[pair[1]][Y.index]
    X = sm.add_constant(X)
    model = sm.OLS(Y, X).fit()
    residuals = model.resid

    # Apply an Augmented Dickey-Fuller test on the residuals:
    adf_result = adfuller(residuals)
    p_value = adf_result[1]

    # Store the p-value into the DataFrame we previously made:
    p_values.loc[pair[0], pair[1]] = p_value
    p_values.loc[pair[1], pair[0]] = p_value

# Output the table of p-values (green highlight: p-value < 0.05, red highlight: p-value > 0.05):
def highlight_significant(val):
    color = 'green' if val < 0.05 else 'red'
    return 'color: %s' % color
styled_p_values = p_values.style.map(highlight_significant)
styled_p_values    



In [ ]:
# Getting the portfolio momentum with the best value 
hor_max = 504
port_mom = compute_mom_method(ret,hor_max)
sr_mom_max = (port_mom.shift()*ret).sum(1)
sr_stat = sr_mom_max.mean()/sr_mom_max.std()*np.sqrt(365*24)

In [ ]:
sr_stat

# Results with Traversal strategy

In [ ]:
# Getting the momentum strategy 
import math

# stats function for later
def get_stats(strat_ret):
    stats = {}
    stats['SR'] = strat_ret.mean() / strat_ret.std() * np.sqrt(365*24)
    stats['ret'] = strat_ret.mean()*365*24
    stats['vol'] = strat_ret.std()*np.sqrt(365*24)
    stats = pd.Series(stats)
    return stats


def compute_traversal_method(ret,hor):
    negate = ret.rolling(hor).mean()*-.1
    pct = negate.rank(axis=1,pct=True)

    long = (pct > 0.9)*1
    long = long.divide(long.abs().sum(1),0)*0.5

    short = (pct < 0.9)*-1.0
    short = short.divide(short.abs().sum(1),0)*0.5

    port = short.add(long,fill_value = 0)
    # port.abs().sum(1).plot()
    # port.sum(1).plot()
    return port
    

strats = {}
xx = 24*365
list_iter = list(range(1,48))
my_new_list = [i * 1 for i in list_iter]
for hor in my_new_list:
    port = compute_traversal_method(ret,hor)
    strat_ret = (port.shift()*ret).sum(1)
    strats[hor] = strat_ret
strats = pd.DataFrame(strats)

port_traversal = compute_traversal_method(ret,1)
sr_tra_max = (port_traversal.shift()*ret).sum(1)
sr_tra_stat = sr_tra_max.mean()/sr_tra_max.std()*np.sqrt(365*24)



In [ ]:
sr = strats.mean()/strats.std()*np.sqrt(365*24)

In [ ]:
sr_tra_stat

## Hybrid approach with both momentum and traversal 

In [ ]:
coeff = 0.65
new_portavg = coeff*port_mom + (1-coeff)*port_traversal

In [ ]:
new_portavg

In [ ]:
 strat_ret_hyb = (new_portavg.shift()*ret).sum(1)

In [ ]:
sr_hybrid = strat_ret_hyb.mean()/strat_ret_hyb.std()*np.sqrt(365*24)

In [ ]:
sr_hybrid

In [ ]:
strat_ret_hyb.cumsum().plot()